# 05 · One cube, six analytical views

## Context

Verbs should be small enough to understand independently and consistent enough
to combine. Every panel answers a real question about the same observations.

## Question

How do summaries, departures, standardization, clipping, and reshaping reveal
different aspects of January minimum temperature?

## Analysis story

Each result begins from the same PRISM cube, so the verb remains the focus.


### Data used in this lesson

Every value comes from the PRISM Group at Oregon State University's AN91d
daily 4 km climate product. This repository carries a small Boulder-region
extract for 1–30 January 2024 so the lesson runs offline without replacing
observations with generated values. The [data validation page](../validation/data.md)
records source URLs, terms, checksums, bounds, units, and acceptance tests.

## Prepare · Use the reviewed minimum-temperature cube

In [ ]:
from pathlib import Path

import xarray as xr

# Find the repository from either a root-level documentation build or a kernel
# started beside this notebook, then open the checksum-controlled PRISM extract.
data_path = next(
    candidate / "tests" / "fixtures" / "real_data" / "prism_boulder_january_2024.nc"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "tests" / "fixtures" / "real_data" / "prism_boulder_january_2024.nc").exists()
)
prism = xr.open_dataset(data_path, engine="scipy").load()

# These assertions are part of the teaching contract: official source,
# canonical cube dimensions, complete daily time, and declared Celsius units.
assert prism.attrs["source"] == "PRISM Group, Oregon State University"
assert prism.attrs["is_synthetic"] == 0
assert prism.sizes == {"time": 30, "y": 24, "x": 24}
assert prism["tmax"].attrs["units"] == "degC"

cube = prism["tmin"]
cube

## Pipes · Build a small, auditable verb gallery

In [ ]:
from cubedynamics import pipe, verbs as v

temporal_mean = (pipe(cube) | v.mean(dim="time")).unwrap()
temporal_variance = (pipe(cube) | v.variance(dim="time")).unwrap()
anomaly = (pipe(cube) | v.anomaly(dim="time")).unwrap()
standardized = (pipe(cube) | v.zscore(dim="time")).unwrap()
bounded = (pipe(standardized) | v.apply(lambda x: x.clip(-2, 2))).unwrap()
flat = (pipe(cube) | v.flatten_cube()).unwrap()

# Flattening changes layout, not the number of observed spatial samples.
assert flat.sizes["sample"] == cube.sizes["y"] * cube.sizes["x"]

## Figure · Compare what each verb preserves and changes

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
temporal_mean.plot(ax=axes[0, 0], cmap="coolwarm", cbar_kwargs={"label": "°C"})
axes[0, 0].set_title("Mean through time")
temporal_variance.plot(ax=axes[0, 1], cmap="viridis", cbar_kwargs={"label": "°C²"})
axes[0, 1].set_title("Variance through time")
anomaly.isel(time=15).plot(ax=axes[0, 2], cmap="RdBu_r", center=0)
axes[0, 2].set_title("Anomaly · 16 January")
standardized.isel(y=12, x=12).plot(ax=axes[1, 0], color="#356d76")
axes[1, 0].set_title("Standardized site history")
bounded.isel(time=15).plot(ax=axes[1, 1], cmap="RdBu_r", vmin=-2, vmax=2)
axes[1, 1].set_title("Custom clipped z-score")
flat.mean("sample").plot(ax=axes[1, 2], color="#6b6544")
axes[1, 2].set_title("Flattened regional mean")
plt.show()

## What the figure tells us

Summary verbs remove dimensions deliberately; anomaly and z-score preserve the
cube; `apply` opens a controlled extension point; flattening changes layout.

## Try the next variation

Change the clipping bound and identify which dates and places are affected.